In [1]:
from selenium.webdriver import Firefox
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support.ui import Select
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.by import By
import xlsxwriter
from time import sleep
import numpy as np
import pandas as pd
from pandas import ExcelFile
from pandas import ExcelWriter  
from selenium import webdriver
from datetime import date
import re
from login import Login
from itertools import product
import numpy as np

In [2]:
# Entra no seges
#nami = webdriver.Chrome(options=options)   
nami = Firefox()
login = "10631094776"
senha = "10631094776"
entrar = Login(nami, login, senha)



The geckodriver version (0.35.0) detected in PATH at C:\ProgramData\chocolatey\bin\geckodriver.exe might not be compatible with the detected firefox version (143.0.4.287); currently, geckodriver 0.36.0 is recommended for firefox 143.*, so it is advised to delete the driver in PATH and retry


In [3]:
#definições

# Verifica se, para o professor “RENATO NUNES DE ANDRADE”, as aulas ministradas são iguais às previstas e diferentes de zero
def aulas_ministradas():
    nami.refresh()
    # espera até o campo estar presente
    WebDriverWait(nami, 10).until(EC.presence_of_element_located((By.TAG_NAME, 'fieldset')))
    
    corpos = nami.find_elements(By.TAG_NAME, 'fieldset')
    linhas = corpos[0].find_elements(By.TAG_NAME, 'tr')
    
    resultado = True  # assume que todos estão "ok" até encontrar um problema
    
    for tr in linhas:
        texto = tr.text
        # só considera linhas com o professor
        if re.search(r"\bRENATO NUNES DE ANDRADE\b", texto, flags=re.IGNORECASE):
            numeros = re.findall(r'\d+', texto)
            if len(numeros) >= 2:
                penultimo, ultimo = int(numeros[-2]), int(numeros[-1])                
                if penultimo != ultimo or (penultimo == 0 and ultimo == 0):
                    resultado = False
                    break
                    
    return resultado


def clica_aguardando_planejamento():
    # pega apenas os textareas com placeholder específico e que estão desabilitados
    elementos = nami.find_elements(
        By.CSS_SELECTOR, 'textarea[placeholder="Aguardando planejamento"][disabled="disabled"]'
    )
    
    for elemento in elementos:
        try:
            elemento.click()
        except Exception:
            pass  # ignora erros caso o elemento não seja clicável



# clica no botão não se aplica
def nao_se_aplica():
    nami.find_elements(By.TAG_NAME, 'fieldset')
    WebDriverWait(nami, 10).until(EC.presence_of_element_located((By.TAG_NAME, 'fieldset')))
    nami.find_elements(By.CSS_SELECTOR, 'select[class = "btn btn-xs"]')
    nami.find_elements(By.CSS_SELECTOR, 'select.btn.btn-xs')
    selects = nami.find_elements(By.CSS_SELECTOR, 'select.btn.btn-xs')

    for s in selects:
        select = Select(s)
        select.select_by_visible_text("Não se aplica")
    

def salva_conteudos():
    # localiza todos os <p> com class="info" que contenham a frase
    paragrafos = nami.find_elements(
        By.XPATH, 
        '//p[@class="info" and contains(., "A RPE precisa ser salva manualmente")]'
    )

    # clica em cada um
    for p in paragrafos:
        try:
            p.click()            
        except Exception as e:
            print("Não foi possível clicar:", e)

## Executavel
def executa(turmas, meses, trimestre):
    for mes, turma in product(meses, turmas):    
        # link para migar o conteudo
        link_padrao = f'https://seges.sedu.es.gov.br/daily_records/{turma}?utf8=%E2%9C%93&by_stage=false&competence={mes}%2F2025&stage_id={trimestre}&empty_lines=0'   
        nami.get(link_padrao)
        
        nami.execute_script("document.body.style.zoom='30%'")
        clica_aguardando_planejamento()
        nao_se_aplica()
        salva_conteudos()
        nami.execute_script("document.body.style.zoom='100%'")
        print(turma, mes, aulas_ministradas())

In [ ]:
meses = np.arange(8, 11, 1)
df = pd.read_excel('id_turma.xlsx')
df.head()
turmas = df.ids
trimestre = 2442
executa(turmas, meses, trimestre)

26233 8 True
26234 8 True
27334 8 True
26235 8 True
26240 8 True
27336 8 True
26242 8 True
26241 8 True
26233 9 True
26234 9 True
27334 9 False
26235 9 False
26240 9 True
27336 9 False
26242 9 False
26241 9 True
26233 10 False
26234 10 False
27334 10 False
26235 10 False
26240 10 False
